In [ ]:
# ===== 全局运行配置 =====
MODE = "demo"          # "eval" 或 "demo"
DATASET = "cnc"        # "cnc" 或 "li"（eval 模式 + demo sample 模式使用）
PROMPT_NAME = "v1"     # prompts 目录下的 prompt 名称，不含 .txt

# demo 模式专用
DEMO_INPUT = "sample"     # "manual"（手动输入单条）或 "sample"（数据集抽样）
DEMO_TEXT = "Heavy rain caused widespread flooding in the region."  # manual 时使用
DEMO_SAMPLE_N = 3        # sample 时使用：取前 N 条

# RAG 开关与模式
USE_RAG = True
RAG_MODE = "knn_pattern"   # "pattern"、"knn" 或 "knn_pattern"
RAG_TOP_K = 5 #每种 retriever 各取 K 条 ,也就是说混合模式为10条

# eval settings
EVAL_SAMPLE_N = 20       # small eval: first N samples
RUN_FULL_EVAL = False    # True runs the full dataset eval
EVAL_PROGRESS_EVERY = 50 # print metric snapshot every N samples

# LLM 参数
TEMPERATURE = 0.0      # eval 用 0 保证可复现；demo 可以调高看多样性
MAX_TOKENS = 2048


In [ ]:
# ===== SPEC_02 验收 =====
# Cell A：导入、环境检查与初始化
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Python executable: {sys.executable}")
if "master_thesis" not in sys.executable.lower():
    raise RuntimeError(
        "当前 Notebook kernel 不是 Master_thesis。请在 Kernel -> Change Kernel 中选择 Master_thesis，"
        "然后 Restart Kernel 后从第一个 cell 重新运行。"
    )

try:
    import openai
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "当前 kernel 缺少 openai，说明 notebook 没有运行在 Master_thesis 环境里，"
        "或 kernel 需要重启。请切换到 Master_thesis 并 Restart Kernel。"
    ) from exc

from src.llm_client import LLMClient
client = LLMClient(temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
print(f"Base URL: {client.base_url}")
print(f"Model: {client.model}")


In [ ]:
# Cell B：连通性测试
assert client.ping(), "LM Studio server 未启动或无法访问"
print("✓ LM Studio 连接正常")


In [ ]:
# Cell C：最简单的 hello 测试
resp = client.chat([{"role": "user", "content": "Reply with exactly the word 'hello' and nothing else."}])
print(f"模型回复: {resp}")


In [ ]:
# Cell D：System message 测试
resp = client.chat([
    {"role": "system", "content": "You are a translator. Translate user input to French."},
    {"role": "user", "content": "Good morning"}
])
print(f"翻译结果: {resp}")


In [ ]:
# Cell E：真实因果抽取试探（只是测 client，不评估输出质量）
resp = client.chat([
    {"role": "user", "content": "Extract cause and effect from this sentence: 'Smoking causes lung cancer.' Output in JSON format."}
])
print(f"因果输出: {resp}")


In [ ]:
# ===== SPEC_03 验收 =====
# Cell F：验证 BGE embedding cache 文件存在
from pathlib import Path

bge_metadata_path = PROJECT_ROOT / "RAG Database" / "bge-small-en-v1.5_examples.jsonl"
bge_embeddings_path = PROJECT_ROOT / "RAG Database" / "bge-small-en-v1.5_embeddings.npy"
assert bge_metadata_path.exists(), f"BGE metadata cache 不存在: {bge_metadata_path}"
assert bge_embeddings_path.exists(), f"BGE embeddings cache 不存在: {bge_embeddings_path}"
print(f"OK BGE metadata cache 已就绪: {bge_metadata_path}")
print(f"OK BGE embeddings cache 已就绪: {bge_embeddings_path}")
print(f"RAG mode: {RAG_MODE} | USE_RAG={USE_RAG} | top_k={RAG_TOP_K}")


In [ ]:
# Cell G：选择 SPEC_03 输入并检验 Prompt 模板渲染（不调 LLM）
import importlib

import src.prompt_builder as prompt_builder
from src.data_io import load_dataset

prompt_builder = importlib.reload(prompt_builder)
build_messages = prompt_builder.build_messages

if DEMO_INPUT == "sample":
    rag_samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    print(f"SPEC_03 sample 预览：{len(rag_samples)} 条（prompt 渲染使用第 1 条）")
    for sample in rag_samples:
        print(f"- id={sample['id']}: {sample['text']}")
    RAG_INPUT_TEXT = rag_samples[0]["text"]
    RAG_INPUT_ID = rag_samples[0]["id"]
else:
    RAG_INPUT_TEXT = DEMO_TEXT
    RAG_INPUT_ID = None

print(f"\nSPEC_03 prompt 输入 id={RAG_INPUT_ID}: {RAG_INPUT_TEXT}")
messages = build_messages(RAG_INPUT_TEXT, use_rag=False, retriever=None, top_k=0, prompt_name=PROMPT_NAME)
for m in messages:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


In [ ]:
# Cell H：按 RAG_MODE 渲染 prompt
import importlib

import src.prompt_builder as prompt_builder
import src.retriever as retriever_module

prompt_builder = importlib.reload(prompt_builder)
retriever_module = importlib.reload(retriever_module)
build_messages = prompt_builder.build_messages
create_retriever = retriever_module.create_retriever

retriever = None
if USE_RAG:
    retriever = create_retriever(
        RAG_MODE,
        metadata_path=bge_metadata_path,
        embeddings_path=bge_embeddings_path,
    )

messages_rag = build_messages(
    RAG_INPUT_TEXT,
    use_rag=USE_RAG,
    retriever=retriever,
    top_k=RAG_TOP_K,
    rag_mode=RAG_MODE,
    prompt_name=PROMPT_NAME,
)
for m in messages_rag:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


In [ ]:
# Cell I：观察当前 RAG_MODE 检索到的 examples
if retriever is None:
    print("RAG 未启用；如需测试检索，请设置 USE_RAG=True，并选择 RAG_MODE。")
else:
    examples = retriever.retrieve(RAG_INPUT_TEXT, top_k=RAG_TOP_K)
    print(f"检索到 {len(examples)} 个 examples:")
    for i, ex in enumerate(examples, 1):
        print(f"\n--- Example {i} ---")
        print(ex)


In [ ]:
# Cell J：用 SPEC_02 的 client 真正调一次 LLM（只看输出，不解析）
resp = client.chat(messages_rag if USE_RAG else messages)
print("=== LLM 原始输出 ===")
print(resp)


In [ ]:
# ===== SPEC_04 验收 =====
# Cell K：parse_output 鲁棒性快速测试（不调 LLM，用构造字符串）
import json
from src.generator import generate, parse_output
from src.data_io import load_dataset

test_cases = [
    '{"has_causal": true, "triples": []}',
    'Here is the result:\n{"has_causal": false, "triples": []}',
    '```json\n{"has_causal": true, "triples": []}\n```',
    'Sure!\n```\n{"has_causal": false, "triples": []}\n```\nDone.',
    '<think>reasoning</think>\n{"has_causal": true, "triples": []}',
]
for i, tc in enumerate(test_cases, 1):
    try:
        result = parse_output(tc)
        print(f"OK 用例 {i}: {result}")
    except Exception as e:
        print(f"FAIL 用例 {i} 解析失败: {e}")


In [ ]:
# Cell L：单条 demo（manual 模式）
if MODE == "demo" and DEMO_INPUT == "manual":
    result = generate(
        text=DEMO_TEXT,
        sample_id=None,
        client=client,
        retriever=retriever if USE_RAG else None,
        use_rag=USE_RAG,
        top_k=RAG_TOP_K,
        rag_mode=RAG_MODE,
        prompt_name=PROMPT_NAME,
    )
    print("=== Demo 单条输出 ===")
    print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# Cell M：数据集抽样 demo（sample 模式）
if MODE == "demo" and DEMO_INPUT == "sample":
    samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    for s in samples:
        result = generate(
            text=s["text"],
            sample_id=s["id"],
            client=client,
            retriever=retriever if USE_RAG else None,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
            prompt_name=PROMPT_NAME,
        )
        print(f"\n--- id={s['id']} ---")
        print(f"输入: {s['text']}")
        print("Gold:")
        print(json.dumps({"has_causal": s["has_causal"], "triples": s["relations"]}, indent=2, ensure_ascii=False))
        print("Pred:")
        print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# ===== SPEC_05 验收 =====
# Cell N：span 预处理与 token-F1 快速检查
from src.evaluator import Evaluator, build_sample_judgement, match_triples, preprocess_span, token_f1

test_pairs = [
    ("A super heavy rain", "super heavy rain"),
    ("The strike resumed on Tuesday.", "strike resumed on tuesday"),
    ("  Multiple   spaces  ", "multiple spaces"),
]
for raw, expected in test_pairs:
    got = preprocess_span(raw)
    mark = "OK" if got == expected else "FAIL"
    print(f"{mark} {raw!r} -> {got!r} (expected {expected!r})")

print(f"exact match: {token_f1('heavy rain', 'heavy rain'):.3f}")
print(f"no overlap: {token_f1('apple pie', 'rocket science'):.3f}")
print(f"article difference: {token_f1('a super heavy rain', 'the super heavy rain'):.3f}")
print(f"partial overlap: {token_f1('heavy rain caused flooding', 'heavy rain'):.3f}")


In [ ]:
# Cell O：构造 mock prediction + gold 跑 evaluator（不调用 LLM）
mock_preds = [
    {"id": 1, "has_causal": True, "triples": [
        {"cause": {"span": "heavy rain"}, "relation": "caused", "effect": {"span": "flooding"}}
    ]},
    {"id": 2, "has_causal": False, "triples": []},
    {"id": 3, "has_causal": True, "triples": [
        {"cause": {"span": "the strike"}, "relation": "caused", "effect": {"span": "delays"}}
    ]},
]
mock_golds = [
    {"id": 1, "has_causal": True, "relations": [{"cause": "heavy rain", "effect": "the flooding"}]},
    {"id": 2, "has_causal": False, "relations": []},
    {"id": 3, "has_causal": False, "relations": []},
]

mock_evaluator = Evaluator()
for pred, gold in zip(mock_preds, mock_golds):
    mock_evaluator.update(prediction=pred, gold=gold)
print(mock_evaluator.format_report(title="SPEC_05 mock evaluator report"))


In [ ]:
# Cell P：流式 eval helper（进度条 + 定期输出 metrics 快照 + 前 10 条明细）
import json
from src.data_io import load_dataset
from src.evaluator import Evaluator, build_sample_judgement
from src.generator import generate
from tqdm.auto import tqdm

def print_debug_judgements(records):
    print("\n================ First 10 sample judgements ================")
    for row in records:
        print(f"\n--- id={row['id']} ---")
        print(f"text: {row['text']}")
        print(f"gold_has_causal={row['gold_has_causal']} | pred_has_causal={row['pred_has_causal']}")
        print(f"token_f1 counts: {row['token_f1']['counts']}")
        print(f"original_like counts: {row['original_like']['counts']}")
        print("gold_relations:")
        print(json.dumps(row["gold_relations"], indent=2, ensure_ascii=False))
        print("pred_triples:")
        print(json.dumps(row["pred_triples"], indent=2, ensure_ascii=False))

def run_stream_eval(samples, label="eval"):
    evaluator = Evaluator()
    debug_records = []
    total = len(samples)
    for i, sample in enumerate(tqdm(samples, total=total, desc=label), 1):
        pred = generate(
            text=sample["text"],
            sample_id=sample["id"],
            client=client,
            retriever=retriever if USE_RAG else None,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
            prompt_name=PROMPT_NAME,
        )
        evaluator.update(prediction=pred, gold=sample)
        if len(debug_records) < 10:
            debug_records.append(build_sample_judgement(prediction=pred, gold=sample))
        if i % EVAL_PROGRESS_EVERY == 0:
            print()
            print(evaluator.format_report(title=f"{label} progress {i}/{total}"))
    print()
    print(evaluator.format_report(title=f"{label} final report"))
    print_debug_judgements(debug_records)
    return evaluator.report()


In [ ]:
# Cell Q：真实数据集小规模 eval（默认前 EVAL_SAMPLE_N 条）
if MODE == "eval":
    eval_samples = load_dataset(DATASET, n=EVAL_SAMPLE_N)
    small_eval_report = run_stream_eval(eval_samples, label=f"{DATASET} first {len(eval_samples)}")
else:
    print("MODE is not eval. Set MODE = 'eval' at the top and rerun the notebook to evaluate real data.")


In [ ]:
# Cell R：完整数据集 eval（默认关闭，耗时较长）
if MODE == "eval" and RUN_FULL_EVAL:
    full_samples = load_dataset(DATASET)
    full_eval_report = run_stream_eval(full_samples, label=f"{DATASET} full")
else:
    print("Full eval did not run. Set MODE = 'eval' and RUN_FULL_EVAL = True to enable it.")
